# 02. Parse Test
이 파일은 확장자별 파싱 로직을 분리해서 관리합니다.

In [ ]:
from pathlib import Path
import sys

WORKDIR = Path.cwd()
REPO_ROOT = WORKDIR.parent if WORKDIR.name == "notebooks" else WORKDIR
if REPO_ROOT.name == "Root_Ingest":
    REPO_ROOT = REPO_ROOT.parent
PROJECT_ROOT = REPO_ROOT / "Root_Ingest"
for path_item in (REPO_ROOT, PROJECT_ROOT):
    if str(path_item) not in sys.path:
        sys.path.append(str(path_item))


In [ ]:
from Root_Ingest.utils.config_loader import load_config
from Root_Ingest.utils.path_utils import resolve_path
from Root_Ingest.ingest.document_loader import collect_documents, load_documents_from_jsonl, save_documents_to_jsonl
from Root_Ingest.ingest.parser_service import parse_documents

config = load_config(PROJECT_ROOT / "config" / "config.yaml")
doc_dir = resolve_path(config["paths"]["doc_dir"], PROJECT_ROOT)
parsed_dir = resolve_path(config["paths"]["parsed_dir"], PROJECT_ROOT)
document_index_path = parsed_dir / "document_index.jsonl"

if document_index_path.exists():
    documents = load_documents_from_jsonl(document_index_path)
else:
    documents = collect_documents(doc_dir, config["document_loader"]["supported_extensions"])
    save_documents_to_jsonl(documents, document_index_path)

parsing_config = config.get("parsing", {})
parser_name = parsing_config.get("parser", "docling")
parser_options = parsing_config.get("options", {})
if not isinstance(parser_options, dict):
    parser_options = {}
legacy_text_encodings = config.get("parser", {}).get("text_encodings")
if legacy_text_encodings and "text_encodings" not in parser_options:
    parser_options["text_encodings"] = legacy_text_encodings

parsed_docs = parse_documents(
    documents=documents,
    output_path=parsed_dir / "parsed_documents.jsonl",
    parser_name=parser_name,
    parser_options=parser_options,
)
print(f"Parsed documents: {len(parsed_docs)}")
parsed_docs[0].raw_text[:500] if parsed_docs else "No parsed output"
